# Kenya Land Readiness Atlas: baseline notebook

Run top to bottom on Colab (CPU runtime is enough, no GPU needed).

What it does: pulls Sentinel-2 features and WorldCover labels for 4 contrasting regions through Earth Engine, caches them to Drive, builds buffered spatial-block folds, trains a shallow decision tree and a LightGBM baseline, evaluates with spatial CV and leave-one-region-out, and reports a confusion matrix, prevalence-weighted accuracy and calibration.

Before running: set PROJECT in the config cell to your Google Cloud project that is registered for Earth Engine.

## 1. Install

In [1]:
!pip -q install -U earthengine-api lightgbm

^C
ERROR: Operation cancelled by user


## 2. Config

In [ ]:
PROJECT = "your-gcp-project-id"   # <-- set this

YEAR = 2021               # WorldCover v200 is a 2021 map, so use 2021 imagery
PER_CLASS = 400           # target samples per class per region (stratified)
CLOUD_THRESH = 0.6        # Cloud Score+ cs_cdf threshold
BLOCK_DEG = 0.05          # spatial block size in degrees (about 5.5 km)
N_FOLDS = 5
SEED = 42
USE_DRIVE = True
OUT_DIR = "/content/drive/MyDrive/land_atlas_baseline" if USE_DRIVE else "/content/land_atlas_baseline"

CLASSES = ["built_up", "cropland", "trees", "grass_shrub", "water", "bare"]

# [min_lon, min_lat, max_lon, max_lat]
REGIONS = {
    "highland":   [36.30, -0.60, 36.80, -0.10],
    "coast":      [39.40, -3.90, 39.90, -3.40],
    "arid":       [37.50,  1.50, 38.00,  2.00],
    "lake_basin": [34.60, -0.30, 35.10,  0.20],
}

## 3. Earth Engine and Drive

In [ ]:
import os, ee
assert PROJECT != "your-gcp-project-id", "Set PROJECT in the config cell first."
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine ready")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)
CACHE = f"{OUT_DIR}/samples_{YEAR}_n{PER_CLASS}.csv"
print("Cache:", CACHE, "| exists:", os.path.exists(CACHE))

## 4. Data: Sentinel-2 features and WorldCover labels

Labels are WorldCover 2021 remapped to 6 classes (wetland to water, mangrove to trees, snow and moss dropped). This measures agreement with WorldCover, not ground truth. The independent hand-labelled test set comes later.

In [ ]:
import time
import numpy as np, pandas as pd

BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
IDX = ["NDVI", "NDWI", "MNDWI", "NDBI"]
EXTRA = ["NDVI_p10", "NDVI_p90", "NDVI_stdDev", "elev", "slope"]
FEATURES = BANDS + IDX + EXTRA
K = len(CLASSES)

def retry(fn, tries=3, wait=10):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            print(f"  retry {i+1}/{tries}: {str(e)[:120]}")
            time.sleep(wait)
    return fn()

def label_image():
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
    # built 0, crop 1, trees 2, grass/shrub 3, water 4, bare 5
    return wc.remap([10, 20, 30, 40, 50, 60, 80, 90, 95],
                    [2, 3, 3, 1, 0, 5, 4, 4, 2]).rename("label").toInt()

def feature_image(geom):
    csp = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterDate(f"{YEAR}-01-01", f"{YEAR+1}-01-01")
          .filterBounds(geom)
          .linkCollection(csp, ["cs_cdf"])
          .map(lambda im: im.updateMask(im.select("cs_cdf").gte(CLOUD_THRESH))))

    def add_idx(im):
        im = im.select(BANDS).divide(10000)
        ndvi = im.normalizedDifference(["B8", "B4"]).rename("NDVI")
        ndwi = im.normalizedDifference(["B3", "B8"]).rename("NDWI")
        mndwi = im.normalizedDifference(["B3", "B11"]).rename("MNDWI")
        ndbi = im.normalizedDifference(["B11", "B8"]).rename("NDBI")
        return im.addBands([ndvi, ndwi, mndwi, ndbi])

    col = s2.map(add_idx)
    med = col.select(BANDS + IDX).median()
    ndvi_stats = col.select("NDVI").reduce(
        ee.Reducer.percentile([10, 90]).combine(ee.Reducer.stdDev(), sharedInputs=True))
    elev = ee.Image("USGS/SRTMGL1_003").rename("elev")
    slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003")).rename("slope")
    return med.addBands(ndvi_stats).addBands(elev).addBands(slope).select(FEATURES)

def pull_region(name, bbox):
    geom = ee.Geometry.Rectangle(bbox)
    img = feature_image(geom).addBands(label_image())
    fc = img.stratifiedSample(
        numPoints=0, classBand="label", region=geom, scale=10, seed=SEED,
        classValues=list(range(K)), classPoints=[PER_CLASS] * K,
        dropNulls=True, tileScale=4, geometries=True)
    feats = retry(lambda: fc.getInfo())["features"]
    rows = [{**f["properties"],
             "lon": f["geometry"]["coordinates"][0],
             "lat": f["geometry"]["coordinates"][1]} for f in feats]
    d = pd.DataFrame(rows)
    d["region"] = name

    # region class shares from WorldCover (coarse 100 m is enough for shares)
    hist = retry(lambda: label_image().reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(), geometry=geom, scale=100,
        maxPixels=int(1e10), bestEffort=True, tileScale=4).get("label").getInfo())
    share = {int(float(k)): v for k, v in hist.items()}
    n = d["label"].value_counts().to_dict()
    present = {c: share.get(c, 0) for c in n}
    tot = sum(present.values())
    # stratified-estimator weight: class share / samples in that class (sums to 1 per region)
    d["weight"] = d["label"].map(lambda c: (present[c] / tot) / n[c])
    return d

if os.path.exists(CACHE):
    df = pd.read_csv(CACHE)
    print("Loaded cache:", df.shape)
else:
    parts = []
    for name, bbox in REGIONS.items():
        t0 = time.time()
        print(f"Sampling {name} ...")
        parts.append(pull_region(name, bbox))
        print(f"  {len(parts[-1])} points in {time.time()-t0:.0f}s")
    df = pd.concat(parts, ignore_index=True)
    df.to_csv(CACHE, index=False)
    print("Saved cache:", CACHE)

df = df.dropna(subset=FEATURES + ["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
print(df.shape)
display(pd.crosstab(df["region"], df["label"].map(dict(enumerate(CLASSES)))))

## 5. Spatial blocks and buffered folds

Points are grouped into 0.05 degree blocks. Whole blocks go to the test fold, and any training point in a block touching a test block is dropped (buffer), so neighbouring pixels cannot leak across the split.

In [ ]:
from sklearn.model_selection import GroupKFold

df["bx"] = np.floor(df["lon"] / BLOCK_DEG).astype(int)
df["by"] = np.floor(df["lat"] / BLOCK_DEG).astype(int)
df["block"] = df["region"] + "_" + df["bx"].astype(str) + "_" + df["by"].astype(str)
print("Blocks:", df["block"].nunique())

reg, bx, by = df["region"].values, df["bx"].values, df["by"].values

def buffered_train(train_idx, test_idx):
    test_blocks = set(zip(reg[test_idx], bx[test_idx], by[test_idx]))
    keep = []
    for i in train_idx:
        near = any((reg[i], bx[i] + dx, by[i] + dy) in test_blocks
                   for dx in (-1, 0, 1) for dy in (-1, 0, 1))
        if not near:
            keep.append(i)
    return np.array(keep)

gkf = GroupKFold(n_splits=N_FOLDS)
FOLDS = []
for tr, te in gkf.split(df, df["label"], groups=df["block"]):
    tr_b = buffered_train(tr, te)
    FOLDS.append((tr_b, te))
    print(f"fold: train {len(tr)} -> {len(tr_b)} after buffer | test {len(te)}")

## 6. Models and metrics helpers

In [ ]:
import lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix

X = df[FEATURES].values
y = df["label"].values
w = df["weight"].values
N = len(df)

LGB_PARAMS = dict(objective="multiclass", n_estimators=300, learning_rate=0.05,
                  num_leaves=31, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
RULE_FEATS = [FEATURES.index(c) for c in ["NDVI", "NDWI", "MNDWI", "NDBI", "elev"]]

def make_model(name):
    if name == "tree_rule":
        return DecisionTreeClassifier(max_depth=4, random_state=SEED)
    return lgb.LGBMClassifier(**LGB_PARAMS)

def cols(name):
    return RULE_FEATS if name == "tree_rule" else list(range(len(FEATURES)))

def fit_predict(name, tr, te):
    m = make_model(name)
    c = cols(name)
    m.fit(X[tr][:, c], y[tr])
    p = np.zeros((len(te), K))
    p[:, m.classes_] = m.predict_proba(X[te][:, c])
    return m, p

def metrics(y_true, proba, weights):
    pred = proba.argmax(1)
    return dict(acc=float((pred == y_true).mean()),
                weighted_acc=float(np.average(pred == y_true, weights=weights)),
                macro_f1=float(f1_score(y_true, pred, average="macro")))

def ece(proba, y_true, bins=10):
    conf, pred = proba.max(1), proba.argmax(1)
    ok = (pred == y_true).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    out, tot = 0.0, len(y_true)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.sum() / tot * abs(ok[m].mean() - conf[m].mean())
    return float(out)

## 7. Spatial block CV (buffered)

In [ ]:
MODELS = ["tree_rule", "lgbm"]
oof = {m: np.full((N, K), np.nan) for m in MODELS}

for name in MODELS:
    for i, (tr, te) in enumerate(FOLDS):
        _, p = fit_predict(name, tr, te)
        oof[name][te] = p
    print("done", name)

rows = []
for name in MODELS:
    r = metrics(y, oof[name], w)
    r["ece"] = ece(oof[name], y)
    r["model"] = name
    rows.append(r)
cv_table = pd.DataFrame(rows).set_index("model").round(4)
display(cv_table)

## 8. Leave-one-region-out

The harder test: train on 3 regions, test on the 4th. Expect lower numbers than block CV. That gap is the honest generalisation story.

In [ ]:
loro_rows = []
for held in REGIONS:
    te = np.where(df["region"].values == held)[0]
    tr = np.where(df["region"].values != held)[0]
    for name in MODELS:
        _, p = fit_predict(name, tr, te)
        r = metrics(y[te], p, w[te])
        r.update(held_out=held, model=name)
        loro_rows.append(r)
loro = pd.DataFrame(loro_rows).set_index(["held_out", "model"]).round(4)
display(loro)
display(loro.groupby("model").mean().round(4).rename(columns=lambda c: c + " (mean)"))

## 9. Confusion matrix, per-region error, calibration

In [ ]:
import matplotlib.pyplot as plt

pred = oof["lgbm"].argmax(1)
cm = confusion_matrix(y, pred, labels=range(K), normalize="true")

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

im = ax[0].imshow(cm, vmin=0, vmax=1, cmap="Blues")
ax[0].set_xticks(range(K)); ax[0].set_xticklabels(CLASSES, rotation=45, ha="right")
ax[0].set_yticks(range(K)); ax[0].set_yticklabels(CLASSES)
for i in range(K):
    for j in range(K):
        ax[0].text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                   color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
ax[0].set_title("LightGBM confusion matrix (row-normalised, block CV)")
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("WorldCover label")

reg_acc = (pd.DataFrame({"region": df["region"], "ok": pred == y, "w": w})
           .groupby("region").apply(lambda g: np.average(g["ok"], weights=g["w"])))
reg_acc.plot.bar(ax=ax[1], color="steelblue")
ax[1].set_ylim(0, 1); ax[1].set_title("Weighted accuracy by region (block CV)")
ax[1].set_ylabel("accuracy")

conf = oof["lgbm"].max(1); ok = (pred == y).astype(float)
edges = np.linspace(0, 1, 11); xs, ys = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.sum() >= 10:
        xs.append(conf[m].mean()); ys.append(ok[m].mean())
ax[2].plot([0, 1], [0, 1], "k--", label="perfect")
ax[2].plot(xs, ys, "o-", label="LightGBM")
ax[2].set_xlabel("mean confidence"); ax[2].set_ylabel("accuracy")
ax[2].set_title(f"Reliability (ECE = {ece(oof['lgbm'], y):.3f})"); ax[2].legend()

plt.tight_layout(); plt.show()

print("Per-class recall (block CV):")
print(pd.Series(np.diag(cm), index=CLASSES).round(3).to_string())

## 10. Feature importance and final model

In [ ]:
final = lgb.LGBMClassifier(**LGB_PARAMS).fit(X, y)
imp = pd.Series(final.booster_.feature_importance("gain"), index=FEATURES).sort_values()
imp.plot.barh(figsize=(6, 7), title="LightGBM gain importance (all data)")
plt.tight_layout(); plt.show()

## 11. Save artefacts and summary

In [ ]:
import json, joblib

final.booster_.save_model(f"{OUT_DIR}/lgbm_baseline.txt")
joblib.dump(final, f"{OUT_DIR}/lgbm_baseline.joblib")

oof_df = df[["region", "block", "lon", "lat", "label"]].copy()
oof_df["pred"] = oof["lgbm"].argmax(1)
oof_df["conf"] = oof["lgbm"].max(1)
oof_df.to_csv(f"{OUT_DIR}/oof_predictions.csv", index=False)

summary = dict(
    year=YEAR, per_class=PER_CLASS, n_points=int(N), classes=CLASSES, features=FEATURES,
    block_cv=cv_table.reset_index().to_dict("records"),
    leave_one_region_out=loro.reset_index().to_dict("records"),
)
with open(f"{OUT_DIR}/baseline_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to", OUT_DIR)
print("\nBlock CV:")
display(cv_table)
print("Leave-one-region-out mean:")
display(loro.groupby("model").mean().round(4))

## Reading these numbers

All accuracy here is agreement with WorldCover, which is itself a model. Weighted accuracy reweights the class-balanced sample by each class's share of the region, so rare classes like built-up do not inflate the headline number. Before you quote any figure, add the independent test: hand-label 300 to 500 stratified random points from high-resolution basemaps and score the model on those with a confusion matrix and area-weighted accuracy (Olofsson et al., 2014).

Next steps from here: swap LightGBM for a ResNet18 or EfficientNet-B0 fine-tune on 64x64 patches, track runs in MLflow, then export the best model to ONNX for CPU serving.

## 12. Dataset Band Normalization & Exploratory Data Analysis (EDA)

Computes exact 10-band dataset-wide statistics across all 8,356 Sentinel-2 patches to refine `_BAND_MEAN` and `_BAND_STD` vectors, and runs EDA to generate class distribution summaries and spectral signature curves.

In [ ]:
# Compute exact 10-band dataset-wide mean and std vectors
!python -m data.compute_band_stats

# Run EDA and feature analysis
!python -m data.eda

## 13. Fine-Tuned PyTorch CNN (Spatial CV & LORO Fine-Tuning)

Trains the fine-tuned Multi-Spectral Sentinel-2 CNN (EfficientNet-B0 / ResNet-18 with Spectral Channel Attention) across 5 spatial block folds and Leave-One-Region-Out (LORO) splits. Exports the best checkpoint to ONNX (`models/land_cover.onnx`).

In [ ]:
!pip install -q onnx onnxscript
!git pull origin main
!python -m training.train --epochs 30